# Grounded RAG Generation

This notebook connects the retrieval system from Day 3 to a language model.

The goal is not simply to generate fluent answers. The model must answer only from retrieved protocol evidence, preserve document and page provenance, cite the evidence it uses, and return `Insufficient Evidence` when the supplied context does not support an answer.

We will keep retrieval and generation separate so that failures can be diagnosed independently.

## 1. Load the project configuration

Day 4 starts from the validated chunks and the persistent Chroma database created earlier.

We will load the chunk dataset, connect to Chroma, load the OpenAI API key, and define the retrieval and generation settings that will be used throughout this notebook.

In [1]:
from pathlib import Path
import os
import re

import pandas as pd
import chromadb

from dotenv import load_dotenv
from openai import OpenAI
from rank_bm25 import BM25Okapi


# Define the paths used by this notebook
chunks_path = Path("../data/processed/protocol_chunks.jsonl")
chroma_path = Path("../data/chroma")
env_path = Path("../.env")


# Load the validated chunks created on Day 2
chunks_df = pd.read_json(
    chunks_path,
    lines=True,
)


# Load the OpenAI API key from the project .env file
load_dotenv(
    env_path,
    override=True,
)

api_key = os.getenv("OPENAI_API_KEY")


# Create the OpenAI client
openai_client = OpenAI(
    api_key=api_key,
)


# Connect to the persistent Chroma database created on Day 3
chroma_client = chromadb.PersistentClient(
    path=str(chroma_path),
)


# Reopen the OpenAI embedding collection
openai_collection = chroma_client.get_collection(
    name="protocol_chunks_openai_v1",
)


# Keep the model choices explicit
embedding_model_name = "text-embedding-3-small"
generation_model_name = "gpt-5.6-terra"


# Define the retrieval settings
semantic_candidate_k = 10
bm25_candidate_k = 10
final_top_k = 5
rrf_k = 60

abstention_message = "Insufficient Evidence"


print("Chunks loaded:", len(chunks_df))
print("Documents:", chunks_df["document_id"].nunique())
print("Chroma records:", openai_collection.count())
print("API key loaded:", api_key is not None)

print("\nEmbedding model:", embedding_model_name)
print("Generation model:", generation_model_name)

print("\nSemantic candidates:", semantic_candidate_k)
print("BM25 candidates:", bm25_candidate_k)
print("Final evidence chunks:", final_top_k)
print("RRF constant:", rrf_k)

Chunks loaded: 172
Documents: 5
Chroma records: 172
API key loaded: True

Embedding model: text-embedding-3-small
Generation model: gpt-5.6-terra

Semantic candidates: 10
BM25 candidates: 10
Final evidence chunks: 5
RRF constant: 60


## 2. Create a reusable hybrid retriever

Day 3 showed that semantic search and BM25 provide different useful retrieval signals.

We will now turn that experimental retrieval logic into one reusable function. The function will search only the selected protocol, retrieve semantic and lexical candidates, combine their rankings using Reciprocal Rank Fusion, and return the highest-ranked evidence chunks.

Later, the LLM will receive only the evidence returned by this function.

In [2]:
def lexical_tokenize(text):
    """Convert text into lowercase word and number tokens."""

    return re.findall(
        r"[A-Za-z0-9]+",
        text.lower(),
    )


def hybrid_retrieve(
    question,
    document_name,
    semantic_k=semantic_candidate_k,
    bm25_k=bm25_candidate_k,
    top_k=final_top_k,
):
    """Retrieve protocol evidence using semantic search, BM25 and RRF."""

    # Keep only chunks belonging to the selected protocol
    document_chunks = (
        chunks_df[
            chunks_df["document_name"] == document_name
        ]
        .copy()
        .reset_index(drop=True)
    )

    # Stop if an unknown protocol name is supplied
    if document_chunks.empty:
        raise ValueError(
            f"Unknown document: {document_name}"
        )


    # Create an OpenAI embedding for the user's question
    query_response = openai_client.embeddings.create(
        model=embedding_model_name,
        input=question,
    )

    query_embedding = query_response.data[0].embedding


    # Do not request more semantic results than the document contains
    semantic_k_actual = min(
        semantic_k,
        len(document_chunks),
    )


    # Retrieve the most semantically similar chunks
    semantic_results = openai_collection.query(
        query_embeddings=[query_embedding],
        n_results=semantic_k_actual,
        where={
            "document_name": document_name
        },
        include=["metadatas"],
    )

    semantic_ids = semantic_results["ids"][0]


    # Record the semantic rank of each returned chunk
    semantic_rank = {
        chunk_id: rank
        for rank, chunk_id in enumerate(
            semantic_ids,
            start=1,
        )
    }


    # Tokenize every chunk for BM25 keyword retrieval
    tokenized_corpus = [
        lexical_tokenize(text)
        for text in document_chunks["text"]
    ]


    # Build the BM25 index for the selected protocol
    bm25 = BM25Okapi(
        tokenized_corpus
    )


    # Tokenize the user's question
    query_tokens = lexical_tokenize(
        question
    )


    # Calculate a BM25 relevance score for every chunk
    bm25_scores = bm25.get_scores(
        query_tokens
    )


    # Keep the highest-ranked BM25 candidates
    bm25_ranked = (
        pd.DataFrame(
            {
                "chunk_id": document_chunks["chunk_id"],
                "bm25_score": bm25_scores,
            }
        )
        .sort_values(
            "bm25_score",
            ascending=False,
        )
        .head(
            min(
                bm25_k,
                len(document_chunks),
            )
        )
    )

    bm25_ids = bm25_ranked[
        "chunk_id"
    ].tolist()


    # Record the BM25 rank of each returned chunk
    bm25_rank = {
        chunk_id: rank
        for rank, chunk_id in enumerate(
            bm25_ids,
            start=1,
        )
    }


    # Combine candidates returned by either retriever
    candidate_ids = set(
        semantic_ids
    ).union(
        bm25_ids
    )

    fused_records = []


    # Calculate the RRF score for every candidate
    for chunk_id in candidate_ids:

        semantic_position = semantic_rank.get(
            chunk_id
        )

        bm25_position = bm25_rank.get(
            chunk_id
        )

        rrf_score = 0.0


        # Add the semantic contribution when available
        if semantic_position is not None:
            rrf_score += 1 / (
                rrf_k + semantic_position
            )


        # Add the BM25 contribution when available
        if bm25_position is not None:
            rrf_score += 1 / (
                rrf_k + bm25_position
            )


        # Find the original chunk and its provenance information
        row = document_chunks[
            document_chunks["chunk_id"] == chunk_id
        ].iloc[0]


        # Store everything needed later by the RAG system
        fused_records.append(
            {
                "chunk_id": chunk_id,
                "document_id": row["document_id"],
                "document_name": row["document_name"],
                "filename": row["filename"],
                "page_number": int(row["page_number"]),
                "chunk_index": int(row["chunk_index"]),
                "text": row["text"],
                "token_count": int(row["token_count"]),
                "rrf_score": rrf_score,
                "semantic_rank": semantic_position,
                "bm25_rank": bm25_position,
            }
        )


    # Rank the combined candidates and keep the strongest evidence
    results_df = (
        pd.DataFrame(fused_records)
        .sort_values(
            "rrf_score",
            ascending=False,
        )
        .head(top_k)
        .reset_index(drop=True)
    )

    return results_df

In [3]:
# Use the difficult INTEGRA question from Day 3
test_question = (
    "What are the participant follow-up time points?"
)

test_document = "INTEGRA"


# Retrieve the best evidence
retrieved_evidence = hybrid_retrieve(
    question=test_question,
    document_name=test_document,
)


# Display only the important retrieval information
print(
    retrieved_evidence[
        [
            "chunk_id",
            "page_number",
            "semantic_rank",
            "bm25_rank",
            "rrf_score",
        ]
    ].to_string(index=False)
)

           chunk_id  page_number  semantic_rank  bm25_rank  rrf_score
PROTO_003_P006_C003            6            1.0        4.0   0.032018
PROTO_003_P005_C002            5            6.0        1.0   0.031545
PROTO_003_P006_C002            6            8.0        2.0   0.030835
PROTO_003_P008_C001            8            4.0        8.0   0.030331
PROTO_003_P007_C003            7            2.0        NaN   0.016129


## 3. Inspect the retrieved evidence

Before sending retrieved chunks to a language model, we should inspect the actual evidence returned by the retriever.

This helps confirm that the retrieval stage has found useful information and prevents us from confusing retrieval problems with language-model problems.

In [4]:
# Display the text of each retrieved chunk
for rank, row in retrieved_evidence.iterrows():

    print(f"Rank {rank + 1}")
    print(f"Chunk ID: {row['chunk_id']}")
    print(f"Protocol: {row['document_name']}")
    print(f"Page: {row['page_number']}")
    print()
    print(row["text"])
    print()

Rank 1
Chunk ID: PROTO_003_P006_C003
Protocol: INTEGRA
Page: 6

(Follow up: 0, 3, 12 m)
Intervention group 1+   n=72
(Follow up:  0, 3, 6, 12 m)
Intervention group 2++   n=72
(Follow up: 0, 3, 12 m)
Control group n= 216
Fig. 1 Study flow chart. *Control group: usual clinical care with the usual control by the family doctor and nurse according to the current CPG
protocol. +Intervention 1: Usual clinical care with the usual control + Monographic consultation + Basic training in clinical practice guidelines +
Training in coaching + 2-h training program to update the coaching strategy + Intervention based on patients SMS phone messages. ++Intervention
2: Usual clinical care with the usual control + Basic training in clinical practice guidelines + Training in coaching + 2-h training program to update t he
coaching strategy + Intervention based on patients SMS phone messages. m: months
Molló et al. BMC Family Practice           (2019) 20:25

Rank 2
Chunk ID: PROTO_003_P005_C002
Protocol: INT

## 4. Build the augmented context

The retriever has identified the most relevant protocol chunks.

Before sending them to the language model, we will format the retrieved evidence into a structured context containing the chunk ID, protocol name, page number, and source text.

Keeping each evidence block clearly labelled will allow the language model to cite its sources and will later allow us to validate those citations automatically.

In [5]:
def format_evidence_context(retrieved_df):
    """Convert retrieved chunks into structured context for the language model."""

    evidence_blocks = []

    # Build one clearly labelled evidence block for each retrieved chunk
    for rank, row in retrieved_df.iterrows():

        block = (
            f"SOURCE {rank + 1}\n"
            f"Chunk ID: {row['chunk_id']}\n"
            f"Protocol: {row['document_name']}\n"
            f"Page: {row['page_number']}\n"
            f"Evidence:\n{row['text']}"
        )

        evidence_blocks.append(block)

    # Separate the evidence blocks with blank lines
    return "\n\n".join(evidence_blocks)


# Format the five retrieved chunks
augmented_context = format_evidence_context(
    retrieved_evidence
)


# Display the beginning of the context
print(augmented_context[:4000])

SOURCE 1
Chunk ID: PROTO_003_P006_C003
Protocol: INTEGRA
Page: 6
Evidence:
(Follow up: 0, 3, 12 m)
Intervention group 1+   n=72
(Follow up:  0, 3, 6, 12 m)
Intervention group 2++   n=72
(Follow up: 0, 3, 12 m)
Control group n= 216
Fig. 1 Study flow chart. *Control group: usual clinical care with the usual control by the family doctor and nurse according to the current CPG
protocol. +Intervention 1: Usual clinical care with the usual control + Monographic consultation + Basic training in clinical practice guidelines +
Training in coaching + 2-h training program to update the coaching strategy + Intervention based on patients SMS phone messages. ++Intervention
2: Usual clinical care with the usual control + Basic training in clinical practice guidelines + Training in coaching + 2-h training program to update t he
coaching strategy + Intervention based on patients SMS phone messages. m: months
Molló et al. BMC Family Practice           (2019) 20:25

SOURCE 2
Chunk ID: PROTO_003_P005_C002


## 5. Create the grounded prompt

The language model should not answer from its own general knowledge.

We will give it clear instructions to use only the retrieved protocol evidence, cite the chunk IDs it relies on, and return `Insufficient Evidence` when the supplied evidence does not support the answer.

This makes the generation stage easier to test and reduces unsupported answers.

In [6]:
def build_grounded_prompt(question, context):
    """Create the instructions and evidence sent to the language model."""

    system_prompt = """
You are a clinical trial protocol document assistant.

Answer the user's question using only the evidence provided to you.

Rules:
1. Do not use outside knowledge.
2. Do not invent or assume missing information.
3. If the evidence does not support the answer, reply exactly: Insufficient Evidence
4. When giving an answer, cite the chunk ID or chunk IDs that support it.
5. Use citations in this exact format: [CHUNK_ID]
6. Preserve important differences between study groups, time points, outcomes, or other protocol details.
7. Keep the answer clear and concise.
""".strip()

    user_prompt = f"""
Question:
{question}

Retrieved evidence:
{context}
""".strip()

    return system_prompt, user_prompt


# Build the prompt for the current INTEGRA question
system_prompt, user_prompt = build_grounded_prompt(
    question=test_question,
    context=augmented_context,
)


print(system_prompt)

print("\nQuestion:")
print(test_question)

You are a clinical trial protocol document assistant.

Answer the user's question using only the evidence provided to you.

Rules:
1. Do not use outside knowledge.
2. Do not invent or assume missing information.
3. If the evidence does not support the answer, reply exactly: Insufficient Evidence
4. When giving an answer, cite the chunk ID or chunk IDs that support it.
5. Use citations in this exact format: [CHUNK_ID]
6. Preserve important differences between study groups, time points, outcomes, or other protocol details.
7. Keep the answer clear and concise.

Question:
What are the participant follow-up time points?


## 6. Generate the first grounded answer

The retrieved evidence and grounding instructions will now be sent to the language model.

This is the generation stage of RAG. The model is expected to answer only from the supplied evidence and cite the exact chunk IDs that support its response.|

In [7]:
# Send the grounded question and retrieved evidence to the language model
response = openai_client.responses.create(
    model=generation_model_name,
    instructions=system_prompt,
    input=user_prompt,
    reasoning={
        "effort": "low"
    },
    text={
        "verbosity": "low"
    },
)


# Extract the generated answer
grounded_answer = response.output_text


print("Question:")
print(test_question)

print("\nAnswer:")
print(grounded_answer)

Question:
What are the participant follow-up time points?

Answer:
- **Intervention group 1:** baseline (0), 3, 6, and 12 months.  
- **Intervention group 2:** baseline (0), 3, and 12 months.  
- **Control group:** 0, 3, and 12 months. [PROTO_003_P006_C003]


## 7. Validate the generated citations

A language model can produce text that looks like a citation even when the referenced source was not supplied to it.

We will therefore validate citations using Python. Every cited chunk ID must exist in the retrieved evidence provided to the model.

This check is deterministic and does not require another language model.

In [8]:
def validate_citations(answer, retrieved_df):
    """Check whether every cited chunk ID came from the retrieved evidence."""

    # Find citations that follow our protocol chunk ID format
    cited_chunk_ids = re.findall(
        r"\[(PROTO_\d+_P\d+_C\d+)\]",
        answer,
    )

    # Get the chunk IDs that were actually supplied to the LLM
    retrieved_chunk_ids = set(
        retrieved_df["chunk_id"].tolist()
    )

    # Find any citations that were not in the retrieved evidence
    invalid_citations = [
        chunk_id
        for chunk_id in cited_chunk_ids
        if chunk_id not in retrieved_chunk_ids
    ]

    return {
        "cited_chunk_ids": cited_chunk_ids,
        "citation_count": len(cited_chunk_ids),
        "invalid_citations": invalid_citations,
        "all_citations_valid": len(invalid_citations) == 0,
    }


citation_check = validate_citations(
    answer=grounded_answer,
    retrieved_df=retrieved_evidence,
)


print("Cited chunks:", citation_check["cited_chunk_ids"])
print("Citation count:", citation_check["citation_count"])
print("Invalid citations:", citation_check["invalid_citations"])
print("All citations valid:", citation_check["all_citations_valid"])

Cited chunks: ['PROTO_003_P006_C003']
Citation count: 1
Invalid citations: []
All citations valid: True


## 8. Inspect the cited evidence

The citation validator confirms that the cited chunk IDs were part of the retrieved evidence.

We should also inspect the actual cited text to confirm that the claims in the generated answer are supported by those sources.

In [9]:
# Get the chunk IDs cited by the language model
cited_chunk_ids = citation_check["cited_chunk_ids"]


# Display only the evidence that the model cited
for chunk_id in cited_chunk_ids:

    row = retrieved_evidence[
        retrieved_evidence["chunk_id"] == chunk_id
    ].iloc[0]

    print("Chunk ID:", row["chunk_id"])
    print("Page:", row["page_number"])
    print()
    print(row["text"])
    print()

Chunk ID: PROTO_003_P006_C003
Page: 6

(Follow up: 0, 3, 12 m)
Intervention group 1+   n=72
(Follow up:  0, 3, 6, 12 m)
Intervention group 2++   n=72
(Follow up: 0, 3, 12 m)
Control group n= 216
Fig. 1 Study flow chart. *Control group: usual clinical care with the usual control by the family doctor and nurse according to the current CPG
protocol. +Intervention 1: Usual clinical care with the usual control + Monographic consultation + Basic training in clinical practice guidelines +
Training in coaching + 2-h training program to update the coaching strategy + Intervention based on patients SMS phone messages. ++Intervention
2: Usual clinical care with the usual control + Basic training in clinical practice guidelines + Training in coaching + 2-h training program to update t he
coaching strategy + Intervention based on patients SMS phone messages. m: months
Molló et al. BMC Family Practice           (2019) 20:25



## 9. Strengthen claim-level grounding

The first generated answer used valid retrieved citations, but inspection showed that one claim relied on ambiguous flowchart extraction.

We will strengthen the generation instructions so that each claim is cited directly, ambiguous evidence is not presented as certain, and the model avoids inferring relationships that are not clearly expressed in the retrieved text.

In [10]:
def build_grounded_prompt(question, context):
    """Create strict evidence-grounded instructions for the language model."""

    system_prompt = """
You are a clinical trial protocol document assistant.

Answer the user's question using only the evidence provided to you.

Rules:
1. Do not use outside knowledge.
2. Do not invent, assume, or infer missing information.
3. If the evidence does not support the answer, reply exactly: Insufficient Evidence
4. Cite every factual claim using the supporting chunk ID.
5. Place the citation directly after the claim it supports.
6. Use citations in this exact format: [CHUNK_ID]
7. Preserve important differences between study groups, time points, outcomes, and protocol details.
8. If extracted table or flowchart text is ambiguous, do not present the ambiguous relationship as certain.
9. Prefer clearly stated prose over uncertain layout-derived interpretations when both are available.
10. Keep the answer concise.
""".strip()

    user_prompt = f"""
Question:
{question}

Retrieved evidence:
{context}
""".strip()

    return system_prompt, user_prompt

In [11]:
system_prompt, user_prompt = build_grounded_prompt(
    question=test_question,
    context=augmented_context,
)

In [12]:
response = openai_client.responses.create(
    model=generation_model_name,
    instructions=system_prompt,
    input=user_prompt,
    reasoning={
        "effort": "low"
    },
    text={
        "verbosity": "low"
    },
)

grounded_answer = response.output_text

print("Question:")
print(test_question)

print("\nAnswer:")
print(grounded_answer)

Question:
What are the participant follow-up time points?

Answer:
- IG-1: baseline (month 0), 3, 6, and 12 months. [PROTO_003_P007_C003]  
- IG-2: baseline (month 0), 3, and 12 months. [PROTO_003_P007_C003]  
- Control group: 0, 3, and 12 months. [PROTO_003_P006_C003]


In [13]:
citation_check = validate_citations(
    answer=grounded_answer,
    retrieved_df=retrieved_evidence,
)

print("Cited chunks:", citation_check["cited_chunk_ids"])
print("Citation count:", citation_check["citation_count"])
print("Invalid citations:", citation_check["invalid_citations"])
print("All citations valid:", citation_check["all_citations_valid"])

Cited chunks: ['PROTO_003_P007_C003', 'PROTO_003_P007_C003', 'PROTO_003_P006_C003']
Citation count: 3
Invalid citations: []
All citations valid: True


## 10. Test an unsupported question

A reliable RAG system should not generate an answer when the retrieved protocol evidence does not contain the requested information.

We will test the abstention behaviour using a question about final trial results. If the supplied evidence does not support an answer, the model should return exactly `Insufficient Evidence`.

In [14]:
unsupported_question = (
    "What were the final results of the INTEGRA trial?"
)


# Retrieve evidence using the same pipeline
unsupported_evidence = hybrid_retrieve(
    question=unsupported_question,
    document_name="INTEGRA",
)


# Format the retrieved chunks for the language model
unsupported_context = format_evidence_context(
    unsupported_evidence
)


# Build the same strict grounded prompt
unsupported_system_prompt, unsupported_user_prompt = build_grounded_prompt(
    question=unsupported_question,
    context=unsupported_context,
)


# Generate the answer
unsupported_response = openai_client.responses.create(
    model=generation_model_name,
    instructions=unsupported_system_prompt,
    input=unsupported_user_prompt,
    reasoning={
        "effort": "low"
    },
    text={
        "verbosity": "low"
    },
)


unsupported_answer = unsupported_response.output_text


print("Question:")
print(unsupported_question)

print("\nAnswer:")
print(unsupported_answer)

Question:
What were the final results of the INTEGRA trial?

Answer:
Insufficient Evidence


## 11. Create the complete RAG pipeline

The individual retrieval, context-formatting, generation, and citation-validation steps are now working.

We will combine them into one reusable function that accepts a question and protocol name, retrieves evidence, generates a grounded answer, validates the citations, and returns the result together with the supporting evidence.

In [15]:
def ask_protocol_question(
    question,
    document_name,
):
    """Run the complete grounded RAG pipeline."""

    # Retrieve the strongest evidence chunks
    evidence = hybrid_retrieve(
        question=question,
        document_name=document_name,
    )


    # Format the retrieved evidence for the language model
    context = format_evidence_context(
        evidence
    )


    # Build the grounding instructions and user prompt
    system_prompt, user_prompt = build_grounded_prompt(
        question=question,
        context=context,
    )


    # Generate the grounded answer
    response = openai_client.responses.create(
        model=generation_model_name,
        instructions=system_prompt,
        input=user_prompt,
        reasoning={
            "effort": "low"
        },
        text={
            "verbosity": "low"
        },
    )


    answer = response.output_text.strip()


    # Validate citations only when the model gives an answer
    if answer == abstention_message :

        citation_validation = {
            "cited_chunk_ids": [],
            "citation_count": 0,
            "has_citations": False,
            "invalid_citations": [],
            "all_citations_valid": True,
        }

    else:

        citation_validation = validate_citations(
            answer=answer,
            retrieved_df=evidence,
        )


    return {
        "question": question,
        "document_name": document_name,
        "answer": answer,
        "citation_validation": citation_validation,
        "evidence": evidence,
    }

## 12. Test the complete pipeline

The complete RAG function will now be tested using the INTEGRA follow-up question.

This verifies that retrieval, evidence formatting, grounded generation, and citation validation work together as one pipeline.

In [16]:
rag_result = ask_protocol_question(
    question="What are the participant follow-up time points?",
    document_name="INTEGRA",
)


print("Question:")
print(rag_result["question"])

print("\nAnswer:")
print(rag_result["answer"])

print("\nCitation validation:")
print(rag_result["citation_validation"])

Question:
What are the participant follow-up time points?

Answer:
- IG-1: baseline (month 0), 3, 6, and 12 months. [PROTO_003_P007_C003]  
- IG-2: baseline (month 0), 3, and 12 months. [PROTO_003_P007_C003]  
- Control group: 0, 3, and 12 months. [PROTO_003_P006_C003]

Citation validation:
{'cited_chunk_ids': ['PROTO_003_P007_C003', 'PROTO_003_P007_C003', 'PROTO_003_P006_C003'], 'citation_count': 3, 'invalid_citations': [], 'all_citations_valid': True}


## 13. Strengthen citation validation

The first citation validator successfully detected invented chunk IDs, but a supported answer with no citations could still pass because there would be no invalid citations to detect.

We will strengthen the validator so that a generated answer must contain at least one valid citation. Abstention responses are handled separately by the complete RAG pipeline.

In [17]:
def validate_citations(answer, retrieved_df):
    """Check that citations exist and refer only to retrieved evidence."""

    # Extract citations that match our chunk ID format
    cited_chunk_ids = re.findall(
        r"\[(PROTO_\d+_P\d+_C\d+)\]",
        answer,
    )

    # Get the chunk IDs that were actually supplied to the LLM
    retrieved_chunk_ids = set(
        retrieved_df["chunk_id"].tolist()
    )

    # Find citations that were not part of the retrieved evidence
    invalid_citations = [
        chunk_id
        for chunk_id in cited_chunk_ids
        if chunk_id not in retrieved_chunk_ids
    ]

    # A normal generated answer should contain at least one citation
    has_citations = len(cited_chunk_ids) > 0

    # Citations pass only when at least one exists and none are invalid
    all_citations_valid = (
        has_citations
        and len(invalid_citations) == 0
    )

    return {
        "cited_chunk_ids": cited_chunk_ids,
        "citation_count": len(cited_chunk_ids),
        "has_citations": has_citations,
        "invalid_citations": invalid_citations,
        "all_citations_valid": all_citations_valid,
    }

In [18]:
citation_check = validate_citations(
    answer=rag_result["answer"],
    retrieved_df=rag_result["evidence"],
)

print(citation_check)

{'cited_chunk_ids': ['PROTO_003_P007_C003', 'PROTO_003_P007_C003', 'PROTO_003_P006_C003'], 'citation_count': 3, 'has_citations': True, 'invalid_citations': [], 'all_citations_valid': True}


## 14. Test abstention through the complete pipeline

The complete RAG function should also handle unsupported questions safely.

We will ask for final INTEGRA trial results again. Because the protocol evidence does not contain those final results, the expected response is `Insufficient Evidence` with no citations.?

In [19]:
unsupported_result = ask_protocol_question(
    question="What were the final results of the INTEGRA trial?",
    document_name="INTEGRA",
)


print("Question:")
print(unsupported_result["question"])

print("\nAnswer:")
print(unsupported_result["answer"])

print("\nCitation validation:")
print(unsupported_result["citation_validation"])

Question:
What were the final results of the INTEGRA trial?

Answer:
Insufficient Evidence

Citation validation:
{'cited_chunk_ids': [], 'citation_count': 0, 'has_citations': False, 'invalid_citations': [], 'all_citations_valid': True}


## Day 4 Findings

The hybrid retrieval system from Day 3 was converted into a reusable retrieval function combining OpenAI semantic search, BM25 lexical retrieval, and Reciprocal Rank Fusion.

Retrieved chunks were formatted into structured evidence containing the protocol name, page number, chunk ID, and source text before being supplied to the language model.

A grounded generation prompt was created requiring the model to answer only from retrieved evidence, cite supporting chunk IDs, and return `Insufficient Evidence` when the evidence does not support an answer.

The supported INTEGRA follow-up question produced grounded answers with valid retrieved citations. Manual inspection also showed that a valid citation does not automatically prove that every claim is perfectly supported when PDF flowchart extraction is ambiguous.

A deterministic citation validator was implemented to check that generated answers contain citations and that every cited chunk ID came from the retrieved evidence.

An unsupported question asking for final INTEGRA trial results correctly returned `Insufficient Evidence`.

The complete workflow was packaged into a reusable `ask_protocol_question()` function that performs retrieval, evidence formatting, grounded generation, abstention handling, and citation validation.

The next stage will evaluate the system using a larger manually labelled question set and use the observed failures to decide whether improvements such as reranking, stronger embeddings, or table-handling changes are justified.